# Análise pré-transformação dos dados

Este notebook é a minha exploração visual para mapear, de forma objetiva, as transformações necessárias antes de consolidar os dados de Dengue, Zika e Chikungunya.
Eu vou verificar estrutura, tipos, valores ausentes e padronizações necessárias, sempre em primeira pessoa e com foco em clareza.

Método (padronizado para todos os dataframes):
- leitura da tabela bruta e preview;
- seleção das colunas de interesse (quando aplicável);
- inspeção de `info()` e `dtypes`;
- análise de valores ausentes por coluna;
- checagens básicas de categorias (ex.: sexo, gestante, sintomas);
- anotações sobre encoding/nomes de colunas e chaves territoriais.

As decisões daqui guiarão os scripts em `src/transform`. No final, eu listo as transformações necessárias para implementar no pipeline.


In [1]:
# Imports mínimos para exploração visual e estrutura
# Obs.: não transformo dados aqui, apenas inspeciono.
import pandas as pd
import numpy as np

In [2]:
# Dengue 2020 — leitura bruta com separador ';' e encoding 'latin1' (bases brasileiras)
# Faço um preview rápido para confirmar estrutura.
df_dengue_2020 = pd.read_csv(
    "../data/raw/dengue/dengon2020_recife.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

df_dengue_2020.head()


,nu_notificacao,tp_notificacao,co_cid,dt_notificacao,ds_semana_notificacao,notificacao_ano,co_uf_notificacao,co_municipio_notificacao,id_regional,co_unidade_notificacao,...,metro,petequias,hematura,sangram,laco_n,plasmatico,evidencia,plaq_menor,con_fhd,complica
0,3545397,2,A90,2020-01-01,202001,2020,26,260790,1497.0,6618464,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3507054,2,A90,2020-01-01,202001,2020,26,261160,1497.0,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3520539,2,A90,2020-01-02,202001,2020,26,261160,1497.0,7958838,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3521380,2,A90,2020-01-03,202001,2020,26,261160,1497.0,6530389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3516940,2,A90,2020-01-03,202001,2020,26,261160,1497.0,2352516,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Colunas de interesse — Dengue (anotações pessoais)

Com base no metadado, vou focar nas seguintes informações sobre os registros:
- número e data da notificação;
- UF e município da notificação;
- idade (derivar de data de nascimento), sexo e gestante;
- UF, município, regional, bairro de residência;
- ocupação (CBO), quando presente;
- sinais e sintomas (febre, mialgia, cefaleia, vômito, náusea etc.).

Essas colunas me permitem relacionar temporalidade, território e condição clínica sem carregar todo o esquema administrativo.


In [3]:
# Subconjunto de colunas que vou analisar para Dengue
# Observação: idade será derivada de 'dt_nascimento' na transformação.
colunas_dengue_interesse = [
    "nu_notificacao",
    "dt_notificacao",
    "co_uf_notificacao",
    "co_municipio_notificacao",
    "dt_nascimento", # Extrair idade por aqui
    "tp_sexo",
    "tp_gestante",
    "co_uf_residencia",
    "co_municipio_residencia",
    "co_bairro_residencia",
    "co_regional_residencia",
    "febre",
    "mialgia",  
    "cefaleia",
    "vomito",
    "nausea",
    "hepatopat",
    "hematolog",
    "diabetes",
    "artrite",
    "conjutivite",
    "dor_costas"
],


In [4]:
# Redução para as colunas de interesse e preview
# Essa cópia evita efeitos colaterais na tabela original.
df_dengue_2020_reduzido = df_dengue_2020[colunas_dengue_interesse].copy()

df_dengue_2020_reduzido.head()


InvalidIndexError: (['nu_notificacao', 'dt_notificacao', 'co_uf_notificacao', 'co_municipio_notificacao', 'dt_nascimento', 'tp_sexo', 'tp_gestante', 'co_uf_residencia', 'co_municipio_residencia', 'co_bairro_residencia', 'co_regional_residencia', 'febre', 'mialgia', 'cefaleia', 'vomito', 'nausea', 'hepatopat', 'hematolog', 'diabetes', 'artrite', 'conjutivite', 'dor_costas'],)

In [ ]:
# Estrutura das colunas e tipos (Dengue reduzido)
df_dengue_2020_reduzido.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3540 entries, 0 to 3539
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   nu_notificacao            3540 non-null   int64  
 1   dt_notificacao            3540 non-null   object 
 2   co_uf_notificacao         3540 non-null   int64  
 3   co_municipio_notificacao  3540 non-null   int64  
 4   dt_nascimento             3488 non-null   object 
 5   tp_sexo                   3540 non-null   object 
 6   tp_gestante               3539 non-null   float64
 7   co_uf_residencia          3540 non-null   int64  
 8   co_municipio_residencia   3540 non-null   int64  
 9   co_bairro_residencia      3529 non-null   float64
 10  co_regional_residencia    3540 non-null   int64  
 11  febre                     3529 non-null   float64
 12  mialgia                   3529 non-null   float64
 13  cefaleia                  3529 non-null   float64
 14  vomito  

In [ ]:
# Distribuições básicas (Dengue): sexo e gestante
# Essas frequências me mostram possíveis inconsistências ou categorias incomuns.
{
    "tp_sexo": df_dengue_2020_reduzido["tp_sexo"].value_counts(dropna=False),
    "tp_gestante": df_dengue_2020_reduzido["tp_gestante"].value_counts(dropna=False),
}

In [ ]:
# Percentual de nulos por coluna (Dengue reduzido)
# Isso me ajuda a priorizar tratamentos.
percentual_nulos_dengue = (
    df_dengue_2020_reduzido.isna().mean().sort_values(ascending=False) * 100
)
percentual_nulos_dengue.round(2)

### Seleção de colunas relevantes — Dengue

A tabela original de dengue possui mais de 120 colunas, muitas delas
administrativas ou específicas para vigilância clínica detalhada.

Para os objetivos deste projeto — análise territorial, demográfica,
temporal e de condições clínicas associadas — foi realizada a seleção
de um subconjunto de colunas consideradas essenciais, conforme definido
a partir do arquivo oficial de metadados.

Essa redução facilita a análise exploratória, melhora desempenho
e torna o pipeline mais claro e reprodutível.


### Escopo da análise estrutural

A análise detalhada de estrutura e possíveis transformações
é realizada inicialmente apenas para o ano de 2020 de Dengue,
por ser representativo do modelo de notificação utilizado
nos anos subsequentes.

Parte-se do pressuposto de que os arquivos de anos posteriores
seguem o mesmo esquema, o que será validado posteriormente
durante a implementação do pipeline de transformação.


### Leitura da tabela bruta — Zika 2021

Aqui eu carrego a tabela de Zika 2021 sem transformações para entender estrutura, tipos e inconsistências. Essa etapa vai orientar a padronização posterior.


In [ ]:
# Zika 2021 — leitura bruta e inspeção de estrutura
df_zika_2021 = pd.read_csv(
    "../data/raw/zika/zika-2021-ok.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

df_zika_2021.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641 entries, 0 to 640
Data columns (total 47 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   NU_NOTIFIC  641 non-null    int64  
 1   TP_NOT      641 non-null    int64  
 2   ID_AGRAVO   641 non-null    object 
 3   CS_SUSPEIT  0 non-null      float64
 4   DT_NOTIFIC  641 non-null    object 
 5   SEM_NOT     641 non-null    int64  
 6   NU_ANO      641 non-null    int64  
 7   SG_UF_NOT   641 non-null    int64  
 8   ID_MUNICIP  641 non-null    int64  
 9   ID_REGIONA  641 non-null    int64  
 10  ID_UNIDADE  641 non-null    int64  
 11  DT_SIN_PRI  641 non-null    object 
 12  SEM_PRI     641 non-null    int64  
 13  DT_NASC     607 non-null    object 
 14  NU_IDADE_N  641 non-null    int64  
 15  CS_SEXO     641 non-null    object 
 16  CS_GESTANT  641 non-null    int64  
 17  CS_RACA     641 non-null    int64  
 18  CS_ESCOL_N  628 non-null    float64
 19  SG_UF       641 non-null    i

### Equivalência de colunas — Zika 2021 (minhas anotações)

Eu verifico correspondências de conceitos para padronizar no pipeline:
| Conceito | Nome no CSV |
|--------|-------------|
| Número da notificação | NU_NOTIFIC |
| Data da notificação | DT_NOTIFIC |
| Ano da notificação | NU_ANO |
| Semana epidemiológica | SEM_NOT |
| Idade (derivar) | DT_NASC |
| Sexo | CS_SEXO |
| Gestante | CS_GESTANT |
| Bairro residência (código/nome) | ID_BAIRRO / NM_BAIRRO |
| Município residência | ID_MN_RESI |
| Regional residência | ID_RG_RESI |


Após a transformação, a tabela de Zika será reduzida para um conjunto
padronizado de colunas compatíveis com Dengue e Chikungunya,
permitindo análises comparativas entre agravos e anos.



In [ ]:
# Subconjunto de colunas para Zika e seleção do dataframe
# Observação: idade será derivada de 'DT_NASC' na transformação.
colunas_zika_interesse = [
    "NU_NOTIFIC",     # Número da notificação
    "DT_NOTIFIC",     # Data da notificação
    "NU_ANO",         # Ano da notificação
    "SEM_NOT",        # Semana epidemiológica
    "SG_UF_NOT",      # UF da notificação
    "ID_MUNICIP",     # Município da notificação
    "DT_NASC",        # Idade [Extrair idade daqui]
    "CS_SEXO",        # Sexo
    "CS_GESTANT",     # Gestante
    "SG_UF",          # UF residência
    "ID_MN_RESI",     # Município residência
    "ID_RG_RESI",     # Regional residência
    "ID_BAIRRO",      # Código do bairro
    "NM_BAIRRO"       # Nome do bairro
]

df_zika_2021_sel = df_zika_2021[colunas_zika_interesse]


In [ ]:
# Preview rápido do subconjunto selecionado (Zika)
df_zika_2021_sel

,NU_NOTIFIC,DT_NOTIFIC,NU_ANO,SEM_NOT,SG_UF_NOT,ID_MUNICIP,DT_NASC,CS_SEXO,CS_GESTANT,SG_UF,ID_MN_RESI,ID_RG_RESI,ID_BAIRRO,NM_BAIRRO
0,3716012,2021-01-11,2021,202102,26,261160,1998-10-02,F,1,26,261160,1497,808,SANTO AMARO
1,3725134,2021-01-27,2021,202104,26,261160,1976-11-25,M,6,26,261160,1497,871,VARZEA
2,3762467,2021-02-04,2021,202105,26,261160,1979-07-20,F,6,26,261160,1497,883,AREIAS
3,3802872,2021-02-10,2021,202106,26,260290,1959-07-06,F,5,26,261160,1497,814,GRACAS
4,3908667,2021-04-28,2021,202117,26,261160,2013-05-19,F,6,26,261160,1497,821,IBURA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,3976982,2021-12-21,2021,202151,26,261160,2001-12-05,F,9,26,261160,1497,867,IPUTINGA
637,3976914,2021-12-01,2021,202148,26,261160,1973-12-03,F,5,26,261160,1497,883,AREIAS
638,4083062,2021-11-09,2021,202145,26,261160,2003-12-30,F,5,26,261160,1497,857,DOIS IRMAOS
639,3976979,2021-12-21,2021,202151,26,261160,2001-06-04,F,1,26,261160,1497,848,APIPUCOS


In [ ]:
# Distribuições básicas (Zika): sexo e gestante
{
    "CS_SEXO": df_zika_2021_sel["CS_SEXO"].value_counts(dropna=False),
    "CS_GESTANT": df_zika_2021_sel["CS_GESTANT"].value_counts(dropna=False),
}

In [ ]:
# Percentual de nulos por coluna (Zika selecionado)
percentual_nulos_zika = (
    df_zika_2021_sel.isna().mean().sort_values(ascending=False) * 100
)
percentual_nulos_zika.round(2)

In [ ]:
# Estrutura das colunas e tipos (Zika selecionado)
df_zika_2021_sel.info()

In [ ]:
# Preview head e estrutura do subconjunto (Zika)
df_zika_2021_sel.head(10)

In [ ]:
# Chikungunya 2021 — leitura bruta
df_chik_2021 = pd.read_csv(
    "../data/raw/chikungunya/chikon-2021-ok.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)


In [ ]:
# Preview direto do dataframe de Chikungunya (cuidado com volume)
df_chik_2021

,ï»¿NU_NOTIFIC,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,...,METRO,PETEQUIAS,HEMATURA,SANGRAM,LACO_N,PLASMATICO,EVIDENCIA,PLAQ_MENOR,CON_FHD,COMPLICA
0,3730663,2,A92.0,01/01/2021,202053,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3730659,2,A92.0,01/01/2021,202053,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3730662,2,A92.0,01/01/2021,202053,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3730664,2,A92.0,02/01/2021,202053,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3730665,2,A92.0,02/01/2021,202053,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17659,4192700,2,A92.0,27/12/2021,202152,2021,26,261160,1497,531,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17660,4087442,2,A92.0,27/12/2021,202152,2021,26,261160,1497,671,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17661,4195237,2,A92.0,27/12/2021,202152,2021,26,261160,1497,604,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17662,4301114,2,A92.0,27/12/2021,202152,2021,26,260790,1497,2319454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Listo todas as colunas para mapear nomes e possíveis ajustes de encoding
df_chik_2021.columns.tolist()

['ï»¿NU_NOTIFIC',
 'TP_NOT',
 'ID_AGRAVO',
 'DT_NOTIFIC',
 'SEM_NOT',
 'NU_ANO',
 'SG_UF_NOT',
 'ID_MUNICIP',
 'ID_REGIONA',
 'ID_UNIDADE',
 'DT_SIN_PRI',
 'SEM_PRI',
 'DT_NASC',
 'NU_IDADE_N',
 'CS_SEXO',
 'CS_GESTANT',
 'CS_RACA',
 'CS_ESCOL_N',
 'SG_UF',
 'ID_MN_RESI',
 'ID_RG_RESI',
 'ID_DISTRIT',
 'ID_BAIRRO',
 'NM_BAIRRO',
 'ID_LOGRADO',
 'NM_LOGRADO',
 'NU_CEP',
 'CS_ZONA',
 'ID_PAIS',
 'DT_INVEST',
 'ID_OCUPA_N',
 'FEBRE',
 'MIALGIA',
 'CEFALEIA',
 'EXANTEMA',
 'VOMITO',
 'NAUSEA',
 'DOR_COSTAS',
 'CONJUNTVIT',
 'ARTRITE',
 'ARTRALGIA',
 'PETEQUIA_N',
 'LEUCOPENIA',
 'LACO',
 'DOR_RETRO',
 'DIABETES',
 'HEMATOLOG',
 'HEPATOPAT',
 'RENAL',
 'HIPERTENSA',
 'ACIDO_PEPT',
 'AUTO_IMUNE',
 'DT_CHIK_S1',
 'DT_CHIK_S2',
 'DT_PRNT',
 'RES_CHIKS1',
 'RES_CHIKS2',
 'RESUL_PRNT',
 'DT_SORO',
 'RESUL_SORO',
 'DT_NS1',
 'RESUL_NS1',
 'DT_VIRAL',
 'RESUL_VI_N',
 'DT_PCR',
 'RESUL_PCR_',
 'SOROTIPO',
 'HISTOPA_N',
 'IMUNOH_N',
 'HOSPITALIZ',
 'DT_INTERNA',
 'UF',
 'MUNICIPIO',
 'TPAUTOCTO',
 '

In [ ]:
# Subconjunto de colunas para Chikungunya
# Observação: nota de BOM em 'NU_NOTIFIC' ("ï»¿NU_NOTIFIC"). Vou tratar isso na transformação.
colunas_chik_interesse = [
    "ï»¿NU_NOTIFIC",
    "DT_NOTIFIC",
    "SG_UF_NOT",
    "ID_MUNICIP",
    "NU_ANO",
    "SEM_NOT",
    "DT_NASC", # extrair idade daqui
    "CS_SEXO",
    "CS_GESTANT",
    "SG_UF",
    "ID_MN_RESI",
    "ID_RG_RESI",
    "ID_BAIRRO",
    "NM_BAIRRO",
    "FEBRE",
    "MIALGIA",
    "CEFALEIA",
    "NAUSEA",
    "VOMITO",
    "ARTRITE",
    "DOR_COSTAS",
    "HEMATOLOG",
    "HEPATOPAT",
    "DIABETES"
]

In [ ]:
# Seleção e preview do subconjunto (Chikungunya)
df_chik_sel =df_chik_2021[colunas_chik_interesse]

df_chik_sel

,ï»¿NU_NOTIFIC,DT_NOTIFIC,SG_UF_NOT,ID_MUNICIP,NU_ANO,SEM_NOT,NU_IDADE_N,CS_SEXO,CS_GESTANT,SG_UF,...,FEBRE,MIALGIA,CEFALEIA,NAUSEA,VOMITO,ARTRITE,DOR_COSTAS,HEMATOLOG,HEPATOPAT,DIABETES
0,3730663,01/01/2021,26,261160,2021,202053,4048,F,9.0,26,...,1,1,2,2,2,2,2,2,2,2
1,3730659,01/01/2021,26,261160,2021,202053,4059,F,9.0,26,...,1,1,2,2,2,2,2,2,2,2
2,3730662,01/01/2021,26,261160,2021,202053,4039,F,9.0,26,...,2,1,2,2,2,2,2,2,2,2
3,3730664,02/01/2021,26,261160,2021,202053,4070,F,9.0,26,...,1,1,1,2,2,2,2,2,2,2
4,3730665,02/01/2021,26,261160,2021,202053,4021,F,9.0,26,...,1,1,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17659,4192700,27/12/2021,26,261160,2021,202152,4025,M,6.0,26,...,1,1,1,2,2,2,2,2,2,2
17660,4087442,27/12/2021,26,261160,2021,202152,4010,M,6.0,26,...,1,1,1,2,2,2,2,2,2,2
17661,4195237,27/12/2021,26,261160,2021,202152,4010,M,6.0,26,...,1,1,1,2,1,2,2,2,2,2
17662,4301114,27/12/2021,26,260790,2021,202152,4026,M,6.0,26,...,1,1,1,2,2,2,2,2,2,2


In [ ]:
# Estrutura das colunas e tipos (Chikungunya selecionado)
df_chik_sel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17664 entries, 0 to 17663
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ï»¿NU_NOTIFIC  17664 non-null  int64  
 1   DT_NOTIFIC     17664 non-null  object 
 2   SG_UF_NOT      17664 non-null  int64  
 3   ID_MUNICIP     17664 non-null  int64  
 4   NU_ANO         17664 non-null  int64  
 5   SEM_NOT        17664 non-null  int64  
 6   NU_IDADE_N     17664 non-null  int64  
 7   CS_SEXO        17664 non-null  object 
 8   CS_GESTANT     17662 non-null  float64
 9   SG_UF          17664 non-null  int64  
 10  ID_MN_RESI     17664 non-null  int64  
 11  ID_RG_RESI     17664 non-null  int64  
 12  ID_BAIRRO      17664 non-null  int64  
 13  NM_BAIRRO      17664 non-null  object 
 14  FEBRE          17664 non-null  int64  
 15  MIALGIA        17664 non-null  int64  
 16  CEFALEIA       17664 non-null  int64  
 17  NAUSEA         17664 non-null  int64  
 18  VOMITO

In [ ]:
# Distribuições básicas (Chikungunya): sexo e gestante
{
    "CS_SEXO": df_chik_sel["CS_SEXO"].value_counts(dropna=False),
    "CS_GESTANT": df_chik_sel["CS_GESTANT"].value_counts(dropna=False),
}

In [ ]:
# Percentual de nulos por coluna (Chikungunya selecionado)
percentual_nulos_chik = (
    df_chik_sel.isna().mean().sort_values(ascending=False) * 100
)
percentual_nulos_chik.round(2)

In [ ]:
# Preview head do subconjunto (Chikungunya)
df_chik_sel.head(10)

## Lista consolidada de transformações necessárias (para o pipeline)

Vou implementar estas transformações no próximo passo (src/transform):
- Padronizar nomes de colunas entre agravos (notificação, datas, residência, sintomas).
- Converter datas para `datetime` (ex.: `dt_notificacao`, `DT_NOTIFIC`, `DT_NASC`) e derivar `idade`.
- Harmonizar categorias de `sexo` e `gestante` (codificação e valores faltantes).
- Tratar BOM/encoding em colunas (ex.: Chikungunya — "ï»¿NU_NOTIFIC").
- Normalizar chaves territoriais (UF, município, bairro, regional) e vincular às tabelas auxiliares.
- Uniformizar nomes de sintomas (caixa alta/baixa) e tipos (bool/int).
- Remover duplicidades e registros inconsistentes; validar intervalo temporal e `SEM_NOT`.
- Filtrar município de interesse (Recife) onde aplicável e documentar critérios.
- Gerar campos derivados padronizados: `ano`, `semana_epi`, `bairro_codigo`, `bairro_nome`.
- Validar e documentar ausências (percentuais de nulos) para decisões de imputação ou descarte.

Com isso, eu garanto comparabilidade entre Dengue, Zika e Chikungunya para análise e dashboards.
